In [1]:
!pip install -q -U google-genai pypdf

from google import genai
from google.genai import types
from google.colab import userdata
import numpy as np

client = genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
print("Setup done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 823.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 22.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.
Setup done.


In [2]:
#STEP-1
from google.colab import files
uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]
print("Uploaded:", pdf_name)

Saving Y23_24_25_B_Tech_BCA_MCA_InSem_Time_Table_2026_2027_ODD_Sem.pdf to Y23_24_25_B_Tech_BCA_MCA_InSem_Time_Table_2026_2027_ODD_Sem.pdf
Uploaded: Y23_24_25_B_Tech_BCA_MCA_InSem_Time_Table_2026_2027_ODD_Sem.pdf


In [3]:
from pypdf import PdfReader

reader = PdfReader(pdf_name)
print("Number of pages:", len(reader.pages))

text = ""
for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        text += extracted + "\n"

print("Total characters extracted:", len(text))

Number of pages: 6
Total characters extracted: 11416


In [4]:
#STEP-2
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(text, chunk_size=800, overlap=150)
print("Number of chunks:", len(chunks))

Number of chunks: 18


In [5]:
print("----- FULL CHUNK 0 -----")
print(chunks[0])
print("----- length:", len(chunks[0]), "characters -----")

----- FULL CHUNK 0 -----
S.No. Date CSE CS&IT ECE
1 07-09-2026
(10:00 AM to 11:30 AM)
22PH1005 - ENGINEERING PHYSICS
23CY1001 - ENGINEERING CHEMISTRY
22PH1005 - ENGINEERING PHYSICS
23CY1001 - ENGINEERING CHEMISTRY 23CY1001-ENGINEERING CHEMISTRY
2 07-09-2026
(02:00PM to 03:30PM)
23CSB3509 - WEB SECURITY
23DEA3505 - MACHINE LEARNING ENGINEERING FOR 
BIG DATA
23DSB3511 - GRAPH AND WEB ANALYTICS
23CEC3510 - EDGE COMPUTING
23AVI3509 - ADVANCED IMAGE PROCESSING AND 
ANALYSIS
23CSB3509 - WEB SECURITY
23DEA3505 - MACHINE LEARNING ENGINEERING FOR BIG 
DATA
23DSB3511 -  GRAPH AND WEB ANALYTICS
23CEC3510 - EDGE COMPUTING
23AVI3509 - ADVANCED IMAGE PROCESSING AND ANALYSIS
22VLS3505-MIXED SIGNAL IC DESIGN;
23EDS3505-EDGE COMPUTING AND DATA 
ANALYTICS IN IOT; 
23WLT3505-MACHINE LEARNING FOR 
WIRELESS COMMUNICATION
08-09-2026
(1
----- length: 800 characters -----


In [6]:
#STEP-3
EMBED_MODEL = "gemini-embedding-001"

def embed_text(t):
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=t,
        config=types.EmbedContentConfig(output_dimensionality=768),
    )
    vec = np.array(result.embeddings[0].values)
    return vec / np.linalg.norm(vec)

chunk_embeddings = []
for i, ch in enumerate(chunks):
    chunk_embeddings.append(embed_text(ch))
    print(f"embedded chunk {i+1}/{len(chunks)}", end="\r")

chunk_embeddings = np.array(chunk_embeddings)
print()
print("Embeddings matrix shape:", chunk_embeddings.shape)

embedded chunk 18/18
Embeddings matrix shape: (18, 768)


In [8]:
#STEP-4
def retrieve(question, k=3):
    q_vec = embed_text(question)
    scores = chunk_embeddings @ q_vec
    top_idx = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top_idx]

In [10]:
#STEP-5
import time
from google.genai.errors import ClientError

test_question = "What time slot is the first exam on 07-09-2026 for BCA students?"

# Adding a simple retry mechanism for rate limits
for attempt in range(3):
    try:
        results = retrieve(test_question, k=3)
        break
    except ClientError as e:
        if "429" in str(e) and attempt < 2:
            print("Rate limit hit, retrying in 5 seconds...")
            time.sleep(5)
        else:
            raise e

for rank, (chunk, score) in enumerate(results, start=1):
    print(f"--- rank {rank} | score: {score:.3f} ---")
    print(chunk)
    print()

--- rank 1 | score: 0.813 ---
urse name. RED colour indicates the examination is MCQ / Skill Review; these courses are conducted in the LAB only.
S.No Date/Time BCA MCA 
1 07-09-2026                    
10.00 AM - 11.30 AM
25SDCA03R - WEB DEVELOPMENT 
USING PYTHON
25SDMC02 - FULL STACK 
APPLICATION DEVELOPMENT
2 07-09-2026               
02.00 PM - 03.30 PM 25UC0008-INDIAN CONSTITUTION 25CA61C2 - PENETRATION TESTING 
& VULNERABILITY ASSESSMENT
3 08-09-2026                
10.00 AM - 11.30 AM
25CA2107-INTRODUCTION TO AI AND 
DATA SCIENCE
25CA6111 - DESIGN & ANALYSIS OF 
ALGORITHMS
4 08-09-2026                       
02.00 PM - 03.30 PM
25MT2101-PROBABILITY AND 
STATISTICS
25CA61C3 - SECURED SOFTWARE 
ENGINEERING
5 09-09-2026               
10.00 AM - 11.30 AM 25CA2105R-DATA STRUCTURES 25CA6112 - CLOUD DEVSECOPS
6 09-09-

--- rank 2 | score: 0.781 ---
rses are conducted in the 
LAB only.
S.No Date BCA 
1 07-09-2026                     10.00 
AM - 11.30 AM
24CA31A5 - DATA VISUALIZATION TE

In [11]:
#STEP-6 & 7
ANSWER_MODEL = "gemini-3.1-flash-lite"

answer_config = types.GenerateContentConfig(
    system_instruction=(
        "You are a question-answering assistant that must rely only on the context "
        "given to you below. If the answer is not contained in the context, reply "
        "exactly: 'I don't know — this isn't covered in the document.' "
        "Never guess or use outside knowledge."
    ),
    temperature=0.2,
)

def ask(question, k=3):
    top = retrieve(question, k)
    context = "\n\n---\n\n".join(chunk for chunk, score in top)

    prompt = f"""Context from the document:

{context}

QUESTION: {question}

Answer using only the context above. If it isn't in the context, say you don't know."""

    response = client.models.generate_content(
        model=ANSWER_MODEL,
        contents=prompt,
        config=answer_config,
    )

    return response.text, top

def ask_and_print(question, k=3):
    answer, top = ask(question, k)
    print("Q:", question)
    print("A:", answer)
    print()
    print("Source chunks used:")
    for rank, (chunk, score) in enumerate(top, start=1):
        preview = chunk[:150].replace("\n", " ")
        print(f"  [{rank}] score {score:.3f} — {preview}...")
    print("=" * 70)

In [13]:
import time
from google.genai.errors import ClientError

in_doc_questions = [
    "Which course code is paired with Engineering Physics for the CSE branch?",
    "On which date and time slot do BCA students have their first exam?",
    "Which course is scheduled for the Data Structures slot for BCA/MCA?",
]

not_in_doc_questions = [
    "What is the passing percentage required in these exams?",
    "What is the reporting time before the exam starts?",
    "Who is the invigilator for the Computer Networks exam?",
]

def ask_with_retry(q, retries=3, delay=5):
    for i in range(retries):
        try:
            ask_and_print(q)
            return
        except ClientError as e:
            if "429" in str(e) and i < retries - 1:
                print(f"Rate limit hit for question: '{q}'. Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                raise e

print("########## IN-DOCUMENT QUESTIONS ##########\n")
for q in in_doc_questions:
    ask_with_retry(q)

print("\n########## NOT-IN-DOCUMENT QUESTIONS ##########\n")
for q in not_in_doc_questions:
    ask_with_retry(q)

########## IN-DOCUMENT QUESTIONS ##########



Q: Which course code is paired with Engineering Physics for the CSE branch?
A: 23CY1001 - ENGINEERING CHEMISTRY

Source chunks used:
  [1] score 0.697 — S.No. Date CSE CS&IT ECE 1 07-09-2026 (10:00 AM to 11:30 AM) 22PH1005 - ENGINEERING PHYSICS 23CY1001 - ENGINEERING CHEMISTRY 22PH1005 - ENGINEERING PH...
  [2] score 0.688 — CS                                23CY1001- ENGINEERING CHEMISTRY                                                                                  22P...
  [3] score 0.636 — S;                                                     24RAN3102A-ADVANCED ROBOTICS;                                     24RAN3102-ADVANCED ROBOTICS; ...
Q: On which date and time slot do BCA students have their first exam?
A: BCA students have their first exam on 07-09-2026 from 10.00 AM - 11.30 AM.

Source chunks used:
  [1] score 0.768 — urse name. RED colour indicates the examination is MCQ / Skill Review; these courses are conducted in the LAB only. S.No Date/Time BCA MCA  1 07-09-20...
  

In [14]:
#CHUNK SIZE-3000
big_chunks = chunk_text(text, chunk_size=3000, overlap=150)
print("Number of chunks (size=3000):", len(big_chunks))

big_chunk_embeddings = []
for i, ch in enumerate(big_chunks):
    big_chunk_embeddings.append(embed_text(ch))
    print(f"embedded chunk {i+1}/{len(big_chunks)}", end="\r")

big_chunk_embeddings = np.array(big_chunk_embeddings)
print()
print("Embeddings matrix shape:", big_chunk_embeddings.shape)

Number of chunks (size=3000): 5
embedded chunk 5/5
Embeddings matrix shape: (5, 768)


In [16]:
import time
from google.genai.errors import ClientError

def retrieve_big(question, k=3):
    q_vec = embed_text(question)
    scores = big_chunk_embeddings @ q_vec
    top_idx = np.argsort(scores)[::-1][:k]
    return [(big_chunks[i], float(scores[i])) for i in top_idx]

def ask_big(question, k=3):
    # Adding a retry loop for rate limits during retrieval
    for attempt in range(3):
        try:
            top = retrieve_big(question, k)
            break
        except ClientError as e:
            if "429" in str(e) and attempt < 2:
                print(f"Rate limit hit during retrieval, retrying in 5 seconds...")
                time.sleep(5)
            else:
                raise e

    context = "\n\n---\n\n".join(chunk for chunk, score in top)

    prompt = f"""Context from the document:

{context}

QUESTION: {question}

Answer using only the context above. If it isn't in the context, say you don't know."""

    response = client.models.generate_content(
        model=ANSWER_MODEL,
        contents=prompt,
        config=answer_config,
    )

    print("Q:", question)
    print("A:", response.text)
    print()
    print("Source chunks used:")
    for rank, (chunk, score) in enumerate(top, start=1):
        preview = chunk[:150].replace("\n", " ")
        print(f"  [{rank}] score {score:.3f} — {preview}...")
    print("=" * 70)

print("########## SAME 3 IN-DOCUMENT QUESTIONS, chunk_size=3000 ##########\n")
for q in in_doc_questions:
    ask_big(q)

########## SAME 3 IN-DOCUMENT QUESTIONS, chunk_size=3000 ##########

Q: Which course code is paired with Engineering Physics for the CSE branch?
A: The course code paired with 22PH1005 - ENGINEERING PHYSICS for the CSE branch is 23CY1001 - ENGINEERING CHEMISTRY.

Source chunks used:
  [1] score 0.656 — S.No. Date CSE CS&IT ECE 1 07-09-2026 (10:00 AM to 11:30 AM) 22PH1005 - ENGINEERING PHYSICS 23CY1001 - ENGINEERING CHEMISTRY 22PH1005 - ENGINEERING PH...
  [2] score 0.638 —                                                                                             23AD2001O - ARTIFICIAL INTELLIGENCE AND MACHINE  LEARNING ...
  [3] score 0.623 — E ENGINEERING 24CI3201-ADAPTIVE SOFTWARE ENGINEERING 24AD3207-NATURAL LANGUAGE PROCESSING  24EC09HF-ADVANCED DIGITAL IC DESIGN 09-09-2026 (02:00PM to ...
Q: On which date and time slot do BCA students have their first exam?
A: BCA students have their first exam on 07-09-2026 from 10.00 AM - 11.30 AM.

Source chunks used:
  [1] score 0.740 — n the

Explanation for changes in chunk size:

Going from chunk size 800 to 3000, the number of chunks dropped from 18 to just 5, because each chunk is now way bigger so fewer are needed to cover the same text. The scores also went down a bit for all three questions, and the top 3 chunks ended up much closer in score to each other. But the answers didn't get worse and all three were still correct, and one even sounded more complete. This probably happened because bigger chunks mix in extra unrelated table rows along with the row I actually need, so the match isn't as sharp anymore, but since my document is small and the rows are close together, the right info was still there either way.